In [ ]:
import functions as fct
import cdsapi
import xarray as xr
import dask

# SISMO

## Download

In [ ]:
# VRG01
fct.download_year(2019)
fct.download_year(2024)

In [ ]:
# PII
fct.download_year(2019,"IV","PII","HHZ")
fct.download_period(2020,1,1,2,29,"IV","PII","HHZ")
fct.download_year(2024,"IV","PII","HHZ")

## Remove response

In [ ]:
fct.remove_response_period("IV","PII","HHZ", y=2019,m1=1,d1=1,m2=12,d2=31)
fct.remove_response_period("IV","PII","HHZ", y=2020, m1=1,d1=1,m2=2,d2=29)
fct.remove_response_period("IV","PII","HHZ", y=2024,m1=1,d1=1,m2=12,d2=31)

## PSD

In [ ]:
# VRG01
all_psd,all_dates,freqs=fct.psd_period(2019,1,1,12,31,station="VRG01")
all_psd,all_dates,freqs=fct.psd_period(2024,1,1,12,31,station="VRG01")

#PII
all_psd,all_dates,freqs=fct.psd_period(2019,1,1,12,31,station="PII",remove_response=True)
all_psd,all_dates,freqs=fct.psd_period(2020,1,1,2,29,station="PII",remove_response=True)
all_psd,all_dates,freqs=fct.psd_period(2024,1,1,12,31,station="PII",remove_response=True)

# METEO

## reanalysis-era5-land
Variables :
- Temperature : ```2m_temperature```
- Wind :```["10m_u_component_of_wind","10m_v_component_of_wind"]```
- Rain : ```"total_precipitation"```
- Surface pressure : ```surface_pressure```

In [ ]:
for m in range(1,13):
    dataset = "reanalysis-era5-land"
    request = {
    "variable": ["2m_temperature"],
    "year": ["2024"],
    "month": [str(m)],
    "day": [str(day) for day in range(1,32)],
    "time": [
    "00:00", "01:00", "02:00", "03:00",
    "04:00", "05:00", "06:00", "07:00",
    "08:00", "09:00", "10:00", "11:00",
    "12:00", "13:00", "14:00", "15:00",
    "16:00", "17:00", "18:00", "19:00",
    "20:00", "21:00", "22:00", "23:00"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [43.7, 10.4, 43.5, 10.6]
    }

    client = cdsapi.Client()


    target = f'virgo_temp_2024_{m}_data.nc'

    client.retrieve(dataset, request, target)

In [ ]:
files = [
    f"virgo_temp_2024_{m}_data.nc"
    for m in range(1, 13)
]

# ouverture et concaténation temporelle
ds = xr.open_mfdataset(
    files,
    combine="by_coords"
)

# sauvegarde en un seul fichier
ds.to_netcdf("virgo_tempp_2024.nc")

## reanalysis-era5-single-levels
- Waves : ```["significant_height_of_total_swell", "significant_height_of_wind_waves"]```

In [ ]:
dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": ["significant_height_of_total_swell", "significant_height_of_wind_waves"],
    "year": ["2024"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [43.6,10.4,43.4,10.2]
}


client = cdsapi.Client()


target = 'virgo_ocean_2024_data.nc'

client.retrieve(dataset, request, target)

# STRAIN

In [ ]:
t0 = 1404143286 #2024-07-04T15:47:48
t1 = t0 + 60*60*24*2 #2 jours

psd, dates, freqs = fct.psd_strain_period(
    "V1",
    t0,
    t1
)
fct.save_psd_strain(
        "psd_strain_1404143286_2j.npz",
        psd,
        dates,
        freqs)

In [ ]:
t0 = 1418140086 #2024-12-13T15:47:48
t1 = t0 + 60*60*24*2 #2 jours

psd_d, dates_d, freqs_d = fct.psd_strain_period(
    "V1",
    t0,
    t1
)
fct.save_psd_strain(
        "psd_strain_1418140086_2j.npz",
        psd_d,
        dates_d,
    
        freqs_d)